
# OpenET API: Effective Precipitation (Peff) for a Polygon

This short notebook shows how to call OpenET's **Peff polygon** endpoint using Python.

**Endpoint used**

`https://california.openet-api.org/experimental/peff/polygon`

**Requirements**: POU geojson & annual report downloaded from the CalWATRS GIS Hub Dashboard: https://cw-gis-waterboards.hub.arcgis.com/apps/calwatrs-public-dashboard/explore,

We'll be using **POU: S021047** for this example

The workflow is:

1. Enter your OpenET API key,
2. Use POU geojson to fetch field IDs, boundaries and properties from the OpenET fields in geodatabase,
3. Define a date range,
4. Request Effective Precipitation from,
5. Load year report from CalWATRS Place of Use,
6. Compare ET of Applied Water (ETaw) with reported diversions

In [ ]:
# Load libraries
import gzip
import json
import requests
import pandas as pd
from io import StringIO
import re
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx
import matplotlib.patheffects as pe


In [ ]:
YOUR_API_KEY = "PASTE_KEY_HERE"
# set your API key before making the request
header = {"Authorization": YOUR_API_KEY}

In [ ]:
# Load GeoJSON downloaded from CalWaters
geojson_path = "./geospatial_data/POU_example.geojson"
with open(geojson_path, "r") as f:
    geojson = json.load(f)

# Get coordinates 
coords = geojson["features"][0]["geometry"]["coordinates"][0]

# To coordinates list:
pou_coords_list = [value for point in coords for value in point]

#### 1. Request fields IDs from OpenET geodatabase in POU

In [ ]:

# OpenET API endpoint: "https://california.openet-api.org/geodatabase/metadata/ids"

ids_response = requests.post(
    url="https://california.openet-api.org/geodatabase/metadata/ids",
    headers=header,
    json={"geometry": pou_coords_list,
        "version": 2.1}
)


In [ ]:
fields_ids_list = list(eval(gzip.decompress(ids_response.content)))
print("OpenET's geodatabase field IDs in POU:", eval(gzip.decompress(ids_response.content)))

#### 2. Get Field Boundaries using the ids

In [ ]:
# OpenET API endpoint: "https://california.openet-api.org/geodatabase/metadata/boundaries"

fields_response = requests.post(
    url="https://california.openet-api.org/geodatabase/metadata/boundaries",
    headers=header,
    json={"field_ids": fields_ids_list}
)


In [ ]:
# Unzip response with the fields GeoJSON
geojson_fields = eval(gzip.decompress(fields_response.content))

In [ ]:
# Plot in WGS84 (EPSG:4326)
pou_gdf = gpd.GeoDataFrame.from_features(
    geojson["features"]
)

fields_gdf = gpd.GeoDataFrame.from_features(
    geojson_fields["features"]
)
fields_gdf = fields_gdf[fields_gdf.geometry.intersects(pou_gdf.geometry.iloc[0])].copy()

fig, ax = plt.subplots(figsize=(10, 10))

# Plot field boundaries
fields_gdf.plot(
    ax=ax,
    color="none",
    edgecolor="darkblue",
    linewidth=3,
    label="OpenET fields"
)

# Plot POU boundary
pou_gdf.plot(
    ax=ax,
    color="none",
    edgecolor="red",
    linewidth=4,
    label="CalWATRS POU"
)

ctx.add_basemap(
        ax,
        source=ctx.providers.Esri.WorldImagery,
        crs="EPSG:4326",
        alpha=0.8
    )

ax.set_axis_off()
ax.set_title("POU and OpenET field boundaries")

from matplotlib.lines import Line2D

legend_handles = [
    Line2D([0], [0], color="darkblue", lw=2, label="OpenET fields"),
    Line2D([0], [0], color="red", lw=3, label="POU"),
]

ax.legend(handles=legend_handles, loc="upper right")
plt.tight_layout()
plt.show()

Keep fields that are fully inside the POU

In [ ]:
pou_geom = pou_gdf.geometry.iloc[0]

overlap_fraction = (
    fields_gdf.geometry.intersection(pou_geom).area
    / fields_gdf.geometry.area
)

# Keep fields with at least 80% area overlap with the POU
fields_gdf = fields_gdf[overlap_fraction >= 0.80].copy()

#### 3. Request properties of fields to get area and crop type

In [ ]:

properties_response = requests.post(
    url="https://california.openet-api.org/geodatabase/metadata/properties",
    headers=header,
    json = {"field_ids": fields_ids_list}
)

In [ ]:
field_properties = pd.DataFrame(eval(gzip.decompress(properties_response.content)))

In [ ]:
# Add acres column
field_properties['acres'] = field_properties['hectares'] * 2.47105

In [ ]:
# Load crop names 
crop_names = pd.read_csv("./tables/cdl_names.csv")
field_properties = field_properties.merge(crop_names, left_on="crop_2024",right_on = "cdl")
field_properties = field_properties.rename(columns={"crop_name":"crop_name_2024"})

In [ ]:
field_properties

In [ ]:
# Merge properties to field boundaries

fields_gdf = fields_gdf.merge(field_properties, on="field_id")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Fill fields by 2024 crop type
fields_gdf.plot(
    ax=ax,
    column="crop_name_2024",
    categorical=True,
    cmap="Greens",
    edgecolor="darkblue",
    linewidth=2,
    alpha=0.65,
    legend=True,
    legend_kwds={
        "title": "Crop type 2024",
        "loc": "upper right"
    }
)

# Plot POU boundary
pou_gdf.plot(
    ax=ax,
    color="none",
    edgecolor="red",
    linewidth=4
)

# Satellite imagery
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    crs="EPSG:4326",
    alpha=0.8
)

ax.set_axis_off()
ax.set_title(
    "POU and selected OpenET field boundaries"
)

plt.tight_layout()
plt.show()

In [ ]:
fields_geojson = fields_gdf

# Keep field_id + geometry
fields_geojson = fields_geojson[["field_id", "geometry"]]

fields_geojson.to_file(
    "geospatial_data/fields.geojson",
    driver="GeoJSON"
)

#### 4. Load selected set of field boundaries in POU to a temporary OpenET asset 

In [ ]:
asset_response = requests.post(
    url="https://california.openet-api.org/account/upload",
    headers=header,
    files={"file": open("geospatial_data/fields.geojson", "rb")}
)

In [ ]:
# Asset ID to use in the Peff request
asset_id = asset_response.json().get("asset_id")
print(asset_id)

#### 5. Effective Precipitation request

In [ ]:
# Define API request parameters

# Water Year 2024
start_date = "2023-10-01"
end_date   = "2024-09-30"

request = {
    "date_range": [start_date, end_date],
    "file_format": "CSV",
    "asset_id": asset_id,
    "attributes": ["field_id"],
    "interval": "monthly",
    "model": "Ensemble",
    "reference_et": "gridMET",
    "units": "in",
    "version": 2.1
}

print(json.dumps(request, indent=2))

Submit Effective Precepitation request by field

In [ ]:

# Request using Peff endpoint "https://california.openet-api.org/experimental/peff/multipolygon"

response = requests.post(
    url="https://california.openet-api.org/experimental/peff/multipolygon",
    headers=header,
    json=request
)

print("HTTP status:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

if response.ok:
    print("Request succeeded.")
    print(response.text[:500])
else:
    print("Request failed.")
    print(response.text)
    response.raise_for_status()


In [ ]:
# Download CSV from response
peff_df = pd.read_csv(StringIO(response.text))

In [ ]:
peff_df["time"] = pd.to_datetime(peff_df["time"])
peff_df["month"] = peff_df["time"].dt.month

In [ ]:
fields_peff = fields_gdf.merge(peff_df,on="field_id",how="left")

In [ ]:
df_field_year = fields_peff.groupby(["field_id","acres","geometry"]).agg({"etaw": "sum"}).reset_index()
# Calculate total volume depth (inches) * area (acres) / 12 to convert to acre-feet
df_field_year["etaw_total_volume"] = df_field_year["etaw"] * df_field_year["acres"]/ 12

Visualize Total ETaw volume by year

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Establish extent
gpd.GeoDataFrame(df_field_year, geometry="geometry").plot(
    ax=ax,
    color="none",
    edgecolor="none"
)

# Satellite basemap
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    crs="EPSG:4326",
    alpha=0.8
)

# Color fields by annual ETaw
gpd.GeoDataFrame(df_field_year, geometry="geometry").plot(
    ax=ax,
    column="etaw_total_volume",
    cmap="viridis_r",
    edgecolor="darkblue",
    linewidth=2,
    alpha=0.75,
    legend=True,
    legend_kwds={"label": "Acre-feet",
                 "orientation": "vertical",
                 "shrink": 0.3,
                 "pad": 0.03}
)

for _, row in fields_gdf.iterrows():

    point = row.geometry.representative_point()

    ax.text(point.x,
            point.y,
            str(row["crop_name_2024"]),
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="white",
            path_effects=[
                pe.withStroke(linewidth=3, foreground="black")
                ]
                )

# POU boundary
pou_gdf.plot(
    ax=ax,
    color="none",
    edgecolor="red",
    linewidth=4
)

ax.set_axis_off()
ax.set_title("Total Annual ETaw Volume by OpenET Field")

plt.tight_layout()
plt.show()

### Load CalWaters Diversions Data

In [ ]:
# Report from CalWATRS Annual report 2024 POU: S021047 
pou_data = pd.read_csv('./tables/annual_reports_-5564496030936987484.csv')

In [ ]:
# Format table to do comparison with OpenET 

row = pou_data.iloc[0]
month_map = {
    "jan": 1,
    "feb": 2,
    "mar": 3,
    "apr": 4,
    "may": 5,
    "jun": 6,
    "jul": 7,
    "aug": 8,
    "sep": 9,
    "oct": 10,
    "nov": 11,
    "dec": 12
}

records = []

# Look through every column
for col in pou_data.columns:

    # Match columns like:
    # wr_janusevolume__c
    # wr_febdiversionvolume__c
    # wr_marstoragevolume__c
    match = re.match(
        r"wr_(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)(.+?)(__c)?$",
        col
    )

    if match:
        month_abbr = match.group(1)
        variable = match.group(2)

        records.append({
            "month": month_map[month_abbr],
            "variable": variable,
            "value": row[col]
        })

monthly_long = pd.DataFrame(records)

In [ ]:
monthly_report = (
    monthly_long
    .pivot(
        index="month",
        columns="variable",
        values="value"
    )
    .reset_index()
)
monthly_report.columns.name = None
monthly_report

Total ETaw volume by month in POU

In [ ]:
df_field_month_pou = fields_peff.groupby(["field_id","month","acres","geometry"]).agg({"etaw": "sum"}).reset_index()
# Calculate total volume depth (inches) * area (acres) / 12 to convert to acre-feet
df_field_month_pou["etaw_monthly_volume"] = df_field_month_pou["etaw"] * df_field_month_pou["acres"]/ 12

In [ ]:
# Monthly total ETaw volume in POU
etaw_pou_month = df_field_month_pou.groupby(["month"]).agg({"etaw_monthly_volume": "sum"}).reset_index()

In [ ]:
etaw_diversions_pou_month = etaw_pou_month.merge(monthly_report, on="month")
etaw_diversions_pou_month.head()

## OpenET ETaw and Reported Diversions Comparison

In [ ]:
# Cumulative sums
# Water year month order: Oct -> Sep
etaw_diversions_pou_month["water_year_month"] = (
   etaw_diversions_pou_month["month"] - 10
) % 12
etaw_diversions_pou_month = (etaw_diversions_pou_month
    .sort_values("water_year_month")
    .reset_index(drop=True)
)

etaw_diversions_pou_month["cum_diversion"] = etaw_diversions_pou_month["diversionvolume"].cumsum()
etaw_diversions_pou_month["cum_etaw"] = etaw_diversions_pou_month["etaw_monthly_volume"].cumsum()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Monthly Diversion vs ETaw plot
axes[0].plot(
    etaw_diversions_pou_month["month"].astype(str),
    etaw_diversions_pou_month["diversionvolume"],
    marker="o",
    label="Reported Diversions"
)

axes[0].plot(
    etaw_diversions_pou_month["month"].astype(str),
    etaw_diversions_pou_month["etaw_monthly_volume"],
    marker="o",
    label="ETaw"
)

axes[0].set_xlabel("Month")
axes[0].set_ylabel("Volume (Acre-Feet)")
axes[0].set_title("Monthly Diversion vs. ETaw")
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.3)


# Cumulative Diversion vs ETaw plot
axes[1].plot(
    etaw_diversions_pou_month["month"].astype(str),
    etaw_diversions_pou_month["cum_diversion"],
    marker="o",
    label="Diversion"
)

axes[1].plot(
    etaw_diversions_pou_month["month"].astype(str),
    etaw_diversions_pou_month["cum_etaw"],
    marker="o",
    label="ETaw"
)

axes[1].set_xlabel("Month")
axes[1].set_title("Cumulative Diversion vs. ETaw")
axes[1].grid(alpha=0.3)

plt.rcParams.update({"font.size": 12})

plt.suptitle("POU: S021047 - Water Year 2024")
plt.tight_layout()
plt.show()